# 04 – Combine niche model with SeaBee remote sensing

Combines the **niche model** (03b) with **SeaBee remote sensing** (shallow water)
into one dataset, which is the input to 05.

- **Niche model**: `nkm-utbredelsesmodell_norge_2026_25833.geo.parquet` – rule-based
  classification from light/salinity/substrate/intertidal.
- **SeaBee remote sensing**:
  `marint_naturkart_habitat_substrat_vegetasjon_regelc_polygoner_uten_rutenett_norge_2024-2025_25833.gpkg`
  – the national regelC habitat product: observed habitat classes
  (ALGAE, ROCK, SEDIMENT, ANGIO, DEEP, ANTHRO) in shallow water.

The rules for how remote sensing overrides the niche model live in
`mnk.rules.classify_seabee` (one rule set, derived from the data model
`nkm_klassifiseringsregler`). ANGIO/ALGAE/ROCK/SEDIMENT are authoritative for
both substrate and light (`nkm_lys`).

**Performance**: the niche model is national (~2.7 M polygons). Only polygons
that actually overlap SeaBee habitat are touched – the rest pass through
unchanged without expensive geometry operations.

In [ ]:
import gc
import io
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import shapely

import mnk
import mnk.rules
import mnk.attributes
import mnk.vectorize
import mnk.utils

CRS = "EPSG:25833"
REPO = Path.cwd()

NICHE_PARQUET = "gs://niva-geodata/MarintNaturKart/results/nkm-utbredelsesmodell_norge_2026_25833.geo.parquet"
SEABEE_GPKG = REPO / "marint_naturkart_habitat_substrat_vegetasjon_regelc_polygoner_uten_rutenett_norge_2024-2025_25833.gpkg"

TORRFALL_PARQUET = "gs://niva-geodata/MarintNaturKart/results/torrfall-filled_norge_25833.geo.parquet"

# Habitat SeaBee classes that can override the niche model.
# DEEP/ANTHRO do not override (see mnk.rules.SEABEE_RULES).
HABITAT_CLASSES = ["ALGAE", "ROCK", "SEDIMENT", "ANGIO"]

# Codes in the niche model (nkm_*) -> tokens in mnk.rules.
def _invert(d):
    return {v: k for k, v in d.items()}

LIGHT_MAP = _invert(mnk.rules.NKM_LYS)
SALINITY_MAP = _invert(mnk.rules.NKM_SALINITET)
SUBSTRATE_MAP = _invert(mnk.rules.NKM_SUBSTRAT)
INTERTIDAL_MAP = _invert(mnk.rules.NKM_TOERFALL)

## 1. Load data

In [ ]:
niche = gpd.read_parquet(NICHE_PARQUET)
assert niche.crs.to_epsg() == 25833
print(f"Niche model: {len(niche):,} polygons")

seabee = gpd.read_file(
    SEABEE_GPKG, layer="polygons", columns=["class_1", "usikkerhet_klasse"]
).to_crs(CRS)
seabee = seabee.rename(columns={"class_1": "klasse"})
seabee["_usikkerhet"] = seabee["usikkerhet_klasse"].replace({"middels": "medium"})
seabee_hab = seabee.loc[
    seabee["klasse"].isin(HABITAT_CLASSES), ["klasse", "_usikkerhet", "geometry"]
].reset_index(drop=True)
print(f"SeaBee habitat: {len(seabee_hab):,} polygons")
print(seabee["klasse"].value_counts())
print(seabee_hab["_usikkerhet"].value_counts())

del seabee
gc.collect()

## 2. SeaBee habitat outside the niche model

The nisjemodell does not reach the whole shallow zone (it clips tørrfall against its
own polygons, and intertidal areas with missing salinity get no polygon).
`seabee_only` is the SeaBee habitat area left after subtracting the nisjemodell
polygons; it is classified in §5. Its salinitet is `brakk` if the SeaBee polygon
touches a nisjemodell polygon with brakk salinitet (`nkm_lm-sa`), else `salt`.

In [ ]:
seabee_only = mnk.vectorize.subtract_layer(seabee_hab, niche, workers=4, progress=True)
seabee_only = seabee_only.explode(index_parts=False).reset_index(drop=True)
seabee_only = seabee_only[~seabee_only.geometry.is_empty & seabee_only.geometry.notna()]

brakk = niche.geometry.values[
    (niche[mnk.rules.COL_SALINITET] == mnk.rules.NKM_SALINITET["brakk"]).to_numpy()
]
b_hit = shapely.STRtree(brakk).query(seabee_only.geometry.values, predicate="intersects")
is_brakk = np.zeros(len(seabee_only), dtype=bool)
is_brakk[np.unique(b_hit[0])] = True
seabee_only["_sal"] = np.where(is_brakk, "brakk", "salt")
print(f"SeaBee-only (no nisjemodell): {len(seabee_only):,} ({is_brakk.sum():,} brakk)")
del brakk, b_hit, is_brakk

## 3. Reclassification of SeaBee pieces

`update_seabee` is applied to every piece under SeaBee habitat (§4 and §5).
`classify_seabee` is resolved once per unique `(klasse, lys, salinitet, substrat,
tørrfall)` combination via `mnk.attributes.map_unique`, then:

- `nin_kode_fkid` / `NiN_name`: from the remote sensing rule; `nkm_prosess` from
  `mnk.attributes.NKM_PROSESS`.
- `nkm_lm-dk`: set to the remotely sensed substrate.
- `nkm_lys`: set from `mnk.rules.SEABEE_LYS_OVERRIDE` (bottom observed ⇒ eufotisk).
- `nkm_sa-fa`: fastsittende-alger basistrinn from the SeaBee class — ALGAE ⇒ `ABC`,
  ROCK ⇒ `0B`; other classes leave it NA. `nkm_seabee_klasse` keeps the class.
- `nkm_usikkerhet_substrat` / `nkm_usikkerhet_lys`: the SeaBee `usikkerhet_klasse`
  (`middels` renamed to `medium`) on the axes the SeaBee class overrides
  (`SEABEE_SUBSTRATE_NKM` / `SEABEE_LYS_NKM`); other axes, and always
  `nkm_usikkerhet_salinitet`, keep the niche model value.
- `nkm_usikkerhetsklasse`: the worst of the three axes
  (`mnk.attributes.worst_usikkerhet`), since salinity still decides MA vs MC.
- `metode_usikkerhet`: `NKM_satelitt_habitat`, plus `|NKM_nisjedata_kvalitet` where
  the niche model's salinity usikkerhet feeds the class.
- `datakilde`: SeaBee is authoritative for lys + substrat (`lys_fjernmaalt_seabee`,
  `substrat_fjernmaalt_seabee`); salinitet stays HI and tørrfall stays kartverket.
- `metode_klassifisering`: the `|`-joined k_metode codes derived from `datakilde`.

In [ ]:
# Remote sensing overrides the model substrate. ANGIO (undervannseng) is soft bottom.
SEABEE_SUBSTRATE_NKM = {
    "ALGAE": mnk.rules.NKM_SUBSTRAT["fast"],
    "ROCK": mnk.rules.NKM_SUBSTRAT["fast"],
    "SEDIMENT": mnk.rules.NKM_SUBSTRAT["løst"],
    "ANGIO": mnk.rules.NKM_SUBSTRAT["løst"],
}
SEABEE_LYS_NKM = {k: mnk.rules.NKM_LYS[v] for k, v in mnk.rules.SEABEE_LYS_OVERRIDE.items()}
SEABEE_SA_FA_NKM = {k: mnk.rules.NKM_SA_FA[v] for k, v in mnk.rules.SEABEE_SA_FA_OVERRIDE.items()}
# SeaBee sets the usikkerhet of the axes it overrides.
SEABEE_USIKKERHET_AXES = {
    "nkm_usikkerhet_substrat": SEABEE_SUBSTRATE_NKM,
    "nkm_usikkerhet_lys": SEABEE_LYS_NKM,
}
USIKKERHET_AXIS_COLS = list(mnk.attributes.USIKKERHET_AXIS_COLS)
METODE_BOTH = mnk.attributes.METODE_SEP.join(
    [mnk.attributes.METODE_USIKKERHET_HABITAT, mnk.attributes.METODE_USIKKERHET_NISJEDATA]
)


def update_seabee(p):
    """Reclassify pieces under SeaBee habitat (columns ``klasse``, ``_usikkerhet``)."""
    keys = pd.DataFrame({
        "klasse": p["klasse"],
        "lys": p[mnk.rules.COL_LYS].map(LIGHT_MAP),
        "sal": p[mnk.rules.COL_SALINITET].map(SALINITY_MAP),
        "sub": p[mnk.rules.COL_SUBSTRAT].map(SUBSTRATE_MAP),
        "tf": p[mnk.rules.COL_TOERFALL].map(INTERTIDAL_MAP),
    })
    nin = mnk.attributes.map_unique(
        keys, lambda *k: mnk.rules.classify_seabee(*k) or (None, None), columns=["kode", "navn"]
    )
    p = p.assign(nin_kode_fkid=nin["kode"].to_numpy(), NiN_name=nin["navn"].to_numpy())
    p = p[p["nin_kode_fkid"].notna()].copy()

    p["nkm_prosess"] = p["nin_kode_fkid"].map(mnk.attributes.NKM_PROSESS)
    p[mnk.rules.COL_SUBSTRAT] = p["klasse"].map(SEABEE_SUBSTRATE_NKM).fillna(p[mnk.rules.COL_SUBSTRAT])
    p[mnk.rules.COL_LYS] = p["klasse"].map(SEABEE_LYS_NKM).fillna(p[mnk.rules.COL_LYS])
    p[mnk.rules.COL_SA_FA] = p["klasse"].map(SEABEE_SA_FA_NKM).astype("string")
    p["nkm_seabee_klasse"] = p["klasse"]

    for col, overrides in SEABEE_USIKKERHET_AXES.items():
        p[col] = p["_usikkerhet"].where(p["klasse"].isin(overrides), p[col]).astype("string")
    p["nkm_usikkerhetsklasse"] = mnk.attributes.worst_usikkerhet(p[USIKKERHET_AXIS_COLS])
    p["metode_usikkerhet"] = np.where(
        p["nkm_usikkerhet_salinitet"].notna(), METODE_BOTH, mnk.attributes.METODE_USIKKERHET_HABITAT
    )

    p["datakilde"] = mnk.attributes.build_datakilde(
        np.full(len(p), mnk.attributes.SOURCE_SUBSTRAT_SEABEE),
        p[mnk.rules.COL_TOERFALL],
        lys_token=mnk.attributes.SOURCE_LYS_SEABEE,
    )
    p["metode_klassifisering"] = mnk.attributes.metode_from_datakilde(p["datakilde"])
    assert p["nkm_prosess"].notna().all()
    return p

## 4. Overlay: SeaBee habitat × niche model

`mnk.vectorize.overlay_update` splits the niche polygons that overlap SeaBee habitat
into the part inside the habitat, which goes through `update_seabee`, and the
remainder, which stays niche model. Only overlapping pairs go through geometry
operations; all other polygons pass through unchanged.

In [ ]:
combined = mnk.vectorize.overlay_update(niche, seabee_hab, update_seabee, workers=4, progress=True)
print(f"Niche ⊕ SeaBee: {len(combined):,} polygons")
print(combined["nkm_seabee_klasse"].value_counts())

del niche, seabee_hab
gc.collect()

## 5. Classify SeaBee habitat with no nisjemodell underneath

For a habitat class the rules need only **salinitet** (§2) and **tørrfall**: `nkm_lys`
is eufotisk and the substrate comes from the SeaBee class. Each polygon starts as
`utenfor` and the part inside the kartverket tørrfall (`TORRFALL_PARQUET`) is set to
`innenfor` with `overlay_update`. Then the same `update_seabee` as in §4 is applied;
salinity usikkerhet is NA, as salinity is inferred from neighbours rather than modelled.

There is **no NA tørrfall canvas** and **no deep-sea canvas**: areas covered by neither
the nisjemodell nor SeaBee stay an honest hole rather than a NA-filled or guessed
polygon.

In [ ]:
so = seabee_only.assign(
    opphav_kode=mnk.attributes.OPPHAV_KODE,
    **{
        mnk.rules.COL_LYS: mnk.rules.NKM_LYS["eufotisk"],
        mnk.rules.COL_SALINITET: seabee_only["_sal"].map(mnk.rules.NKM_SALINITET),
        mnk.rules.COL_SUBSTRAT: mnk.rules.NKM_SUBSTRAT["løst"],
        mnk.rules.COL_TOERFALL: mnk.rules.NKM_TOERFALL["utenfor"],
    },
)
for col in USIKKERHET_AXIS_COLS:
    so[col] = pd.Series(pd.NA, index=so.index, dtype="string")

torrfall = gpd.read_parquet(TORRFALL_PARQUET).to_crs(CRS)[["geometry"]].reset_index(drop=True)
torrfall["geometry"] = shapely.make_valid(torrfall.geometry.to_numpy())
so = mnk.vectorize.overlay_update(
    so, torrfall, lambda p: p.assign(**{mnk.rules.COL_TOERFALL: mnk.rules.NKM_TOERFALL["innenfor"]})
)
del torrfall
gc.collect()

so = update_seabee(so)
innenfor = (so[mnk.rules.COL_TOERFALL] == mnk.rules.NKM_TOERFALL["innenfor"]).sum()
print(f"SeaBee-only pieces: {len(so):,} ({innenfor:,} innenfor tørrfall)")

## 6. Combine

`final = niche ⊕ SeaBee (§4) + SeaBee-only pieces (§5)`. Working columns (`_*`, the
SeaBee `klasse`, `nin_int`) are dropped; `nkm_seabee_klasse` and `nkm_sa-fa` are NA
for the pure nisjemodell pieces.

`metode_klassifisering` is (re)derived here for every classified row from its
`datakilde` via `mnk.attributes.metode_from_datakilde` — the `|`-joined k_metode
codes of the contributing sources (e.g. `NKM_modell_lyssone|NKM_modell_salinitet|
NKM_modell_bunntype`).

In [ ]:
final = gpd.GeoDataFrame(pd.concat([combined, so], ignore_index=True), crs=CRS)
final = final[[c for c in final.columns if not c.startswith("_") and c not in ("klasse", "nin_int", "nk_fid")]]

classified = final["nin_kode_fkid"].notna()
final.loc[classified, "metode_klassifisering"] = mnk.attributes.metode_from_datakilde(
    final.loc[classified, "datakilde"]
)

print(f"Combined (niche ⊕ SeaBee): {len(final):,} polygons ({len(so):,} SeaBee-only)")
print(final["nin_kode_fkid"].value_counts())
print(final["metode_klassifisering"].value_counts())

del combined, seabee_only, so
gc.collect()

## 7. Save

In [ ]:
fname = mnk.utils.to_filename("nisjemodel-remote-sensing", "norge", "2026", CRS.split(":")[1])
parquet_path = REPO / f"{fname}.geo.parquet"

obj_cols = final.select_dtypes("object").columns
final[obj_cols] = final[obj_cols].astype("string")

_meta = io.BytesIO()
final.iloc[:0].to_parquet(_meta)
_meta.seek(0)
geo_meta = pq.read_schema(pa.PythonFile(_meta, mode="r")).metadata[b"geo"]
schema = pa.table(final.iloc[:0].to_arrow(geometry_encoding="WKB")).schema.with_metadata(
    {b"geo": geo_meta}
)

CHUNK = 100_000
with pq.ParquetWriter(parquet_path, schema, compression="snappy") as w:
    for start in range(0, len(final), CHUNK):
        part = pa.table(final.iloc[start : start + CHUNK].to_arrow(geometry_encoding="WKB"))
        w.write_table(part.cast(schema))
        del part
        gc.collect()
print(f"Saved {len(final):,} polygons -> {parquet_path.name}")